# The Point Spread Function


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-17.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

A perfect lens does not image a point source to a point. Diffraction at the rim of
the aperture spreads the light into a bright central disk surrounded by faint rings.
That pattern is the **point spread function**, and it is the single object that
decides how sharp an optical system can ever be.

This notebook builds it from the chapter's expression

$$h(\rho) = h(0)\,\frac{2 J_1\!\left(\pi D \rho / \lambda f\right)}{\pi D \rho / \lambda f}$$

for a circular aperture of diameter $D$ and focal length $f$, and then shows the two
facts the convolution integral depends on: the PSF has a **fixed width** set by
$\lambda f / D$, and it has a **fixed shape** that simply moves to wherever the source
images to.


## Import libraries

`scipy.special.j1` is the first-order Bessel function $J_1$. It is the only piece of
machinery here that is not in NumPy, and it is what makes the Airy pattern a formula
rather than a numerical diffraction calculation.

`Circle` draws the first dark ring on top of the 2D pattern.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.special import j1


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**Real numbers, not arbitrary units.** A 25 mm lens of 100 mm focal length in green
light is an ordinary bench setup, and it fixes the one length scale in the problem:

$$\frac{\lambda f}{D} = \frac{(0.55~\mu\mathrm{m})(100~\mathrm{mm})}{25~\mathrm{mm}}
= 2.2~\mu\mathrm{m}$$

Everything about the PSF is this number times a pure constant. The first dark ring sits
at $1.22\,\lambda f / D$, so this system cannot concentrate light into a spot smaller
than about $2.7~\mu$m in radius no matter how well the lens is made.

**`jinc` handles the removable singularity.** The ratio $2J_1(u)/u$ is perfectly finite
at $u = 0$, where it equals 1, but evaluating it there divides zero by zero. Substituting
a tiny value for $u$ avoids the warning without measurably changing the curve.

**Amplitude and irradiance are different things.** $h$ is the field amplitude and can go
negative; $\left\lvert h \right\rvert^2$ is what a detector measures, and it is a **magnitude**
squared because $h$ is complex in general. **Coherent** imaging convolves the object
with $h$; **incoherent** imaging convolves it with $\left\lvert h \right\rvert^2$, so both are computed here.


In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
# All lengths in micrometers.
lam1 = 0.55        # wavelength, green light
f_lens1 = 1.0e5    # focal length, 100 mm
diam1 = 2.5e4      # lens diameter, 25 mm

scale1 = lam1 * f_lens1 / diam1        # lambda f / D, the only length scale
r_airy1 = 1.2196699 * scale1           # radius of the first dark ring


def jinc(u):
    """2 J1(u) / u, which equals 1 at u = 0.

    The ratio is finite at the origin but 0/0 numerically, so substitute a
    tiny argument there instead of dividing by zero.
    """
    u_safe = np.where(np.abs(u) < 1e-12, 1e-12, u)
    return 2 * j1(u_safe) / u_safe


def psf_amplitude(rho):
    """Field amplitude h of the Airy pattern, normalized to h(0) = 1."""
    return jinc(np.pi * diam1 * rho / (lam1 * f_lens1))


# --- Radial cut through the pattern ---
rho1 = np.linspace(-8.0, 8.0, 4001)
h_amp1 = psf_amplitude(rho1)
h_int1 = h_amp1 ** 2                   # what a detector records

# --- The image plane, as a 2D map ---
half1 = 10.0
n_grid1 = 601
X1_axis = np.linspace(-half1, half1, n_grid1)
X1, Y1 = np.meshgrid(X1_axis, X1_axis)

X_src1, Y_src1 = 3.0, 2.5               # where the source images to, M*x0 and M*y0
rho_2d1 = np.sqrt((X1 - X_src1) ** 2 + (Y1 - Y_src1) ** 2)
h_int_2d1 = psf_amplitude(rho_2d1) ** 2

gamma1 = 0.4                            # display stretch, so the rings are visible


## Plot the point spread function

The left panel is a cut straight through the pattern, the right panel is the pattern
itself.

**In the cut**, notice how much narrower the irradiance is than the amplitude. Squaring
a function that is already peaked at the center pulls it in further and crushes the
sidelobes: the first ring holds only **1.75%** of the peak. That is why the rings are
easy to miss on a linear scale, and why they still matter when a faint object sits next
to a bright one.

**In the image plane** the same faintness is the problem. Displaying $\left\lvert h \right\rvert^2$ directly
would give a white dot on a black field and nothing else, so the panel plots
$\left(\left\lvert h \right\rvert^2\right)^{0.4}$, which is the same compression a camera applies when it
stretches a raw exposure. The rings are real, and this is what it takes to see them.

The dashed circle is the first dark ring at $1.22\,\lambda f / D$. Everything inside it
carries **83.78%** of the light, which is the usual justification for treating the
Airy disk as if it were the whole spot.


In [ ]:
# ==========================================
# 3. Plotting the PSF
# ==========================================
fig1, (ax_cut1, ax_img1) = plt.subplots(
    1, 2, figsize=(15, 6.2), layout='constrained',
    gridspec_kw={'width_ratios': [1.25, 1]})

# --- Left: the radial cut ---
ax_cut1.plot(rho1, h_amp1, 'b--', label=r'amplitude  $h(\rho)$')
ax_cut1.plot(rho1, h_int1, 'r-', label=r'irradiance  $\left| h(\rho) \right|^2$')
ax_cut1.fill_between(rho1, 0, h_int1, color='red', alpha=0.12)
ax_cut1.axhline(0, color='black', linewidth=1, alpha=0.5)

# The first dark ring, on both sides of the axis
for sign1 in (-1, +1):
    ax_cut1.plot(sign1 * r_airy1, 0, 'ko', markersize=7, zorder=5)
# Label the left-hand zero, so this annotation and the sidelobe one below sit on
# opposite sides of the peak instead of crowding each other
ax_cut1.annotate(r"$1.22\,\lambda f / D = %.2f\ \mu$m" % r_airy1,
                 xy=(-r_airy1, 0.0), xytext=(-4.9, 0.47),
                 arrowprops=dict(arrowstyle="->", color='black', linewidth=1.8),
                 fontsize=15, ha='center')

# The first ring is easy to overlook, so say how small it is.
# Its peak sits at 1.635 lambda f / D, just outside the first zero.
ax_cut1.annotate("first ring,\n1.75% of peak",
                 xy=(1.635 * scale1, 0.0175), xytext=(6.2, 0.20),
                 arrowprops=dict(arrowstyle="->", color='dimgray', linewidth=1.5),
                 fontsize=13, color='dimgray', ha='center')

ax_cut1.set_title("Cross-section through the pattern", fontsize=16)
ax_cut1.set_xlabel(r"radial distance $\rho$  ($\mu$m)")
ax_cut1.set_ylabel("normalized value")
ax_cut1.set_xlim(-8, 8)
ax_cut1.set_ylim(-0.15, 1.12)
ax_cut1.legend(loc='upper left', frameon=True, edgecolor='black',
               fancybox=False, fontsize=13)
apply_custom_formatting(ax_cut1)

# --- Right: the image plane ---
ax_img1.imshow(h_int_2d1 ** gamma1, extent=[-half1, half1, -half1, half1],
               cmap='gray', origin='lower', vmin=0, vmax=1)

# Crosshairs locating the spot on the two axes
ax_img1.plot([-half1, X_src1], [Y_src1, Y_src1], color='deepskyblue',
             linestyle=':', linewidth=1.5)
ax_img1.plot([X_src1, X_src1], [-half1, Y_src1], color='deepskyblue',
             linestyle=':', linewidth=1.5)

# The first dark ring
ax_img1.add_patch(Circle((X_src1, Y_src1), r_airy1, fill=False,
                         edgecolor='yellow', linestyle='--', linewidth=1.6))
ax_img1.annotate(r"$1.22\,\lambda f / D$",
                 xy=(X_src1 + 0.71 * r_airy1, Y_src1 + 0.71 * r_airy1),
                 xytext=(X_src1 + 2.2, Y_src1 + 4.7),
                 arrowprops=dict(arrowstyle="->", color='yellow', linewidth=1.6),
                 fontsize=14, color='yellow', ha='center')

ax_img1.text(X_src1 + 0.5, Y_src1 - 3.1, r"$(M x_0,\ M y_0)$",
             fontsize=15, color='deepskyblue', ha='left', va='top')

ax_img1.set_title("Image of a single point source", fontsize=16)
ax_img1.set_xlabel(r"$X$  ($\mu$m)")
ax_img1.set_ylabel(r"$Y$  ($\mu$m)")
ax_img1.tick_params(which='major', direction='in', length=6, width=1,
                    color='white', top=True, right=True)

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig1.savefig("../figures/ch05/ch05-17-psf-airy.jpg", dpi=200, bbox_inches='tight')


## Where the PSF sits

The panel above put the source on the axis. Move the source, and the pattern moves with
it, unchanged in width and shape. That property has a name, **shift invariance**, and
without it the rest of the chapter would not work.

Here is why it matters. A source at object position $y_0$ images to $Y_0 = M y_0$, and
the light it contributes to some other image point $Y$ is the incoherent PSF
evaluated at the **separation** between them, $\left\lvert h(Y - Y_0) \right\rvert^2$. Not at $Y$, and not at $Y_0$, but at the
difference. The marked point on the curve is that one number.

An extended object is a continuum of such sources, so the image at $Y$ is the sum of
$\left\lvert h(Y - Y_0) \right\rvert^2$ over every source, weighted by how bright each one is. Written as an
integral that sum is

$$I_\textrm{image}(Y) = \int I_\textrm{obj}(y)\, \left\lvert h(Y - M y) \right\rvert^2\, dy$$

which is the convolution the next section of the chapter derives. The dependence on
$Y - Y_0$ alone, visible in this figure as one shape sliding without distorting, is
exactly the condition that makes it a convolution rather than a general integral
transform.


In [ ]:
# ==========================================
# 4. Plotting the shift
# ==========================================
Y_shift1 = 5.0      # Y_0, where the source images to
# The image point being asked about. Put it on the flank of the PSF rather than
# out in a dark ring, so the value being read off is visibly non-zero.
Y_obs1 = 6.4

Y_line1 = np.linspace(-12.0, 16.0, 6001)
psf_on_axis1 = psf_amplitude(Y_line1) ** 2
psf_shifted1 = psf_amplitude(Y_line1 - Y_shift1) ** 2
h_sample1 = float(psf_amplitude(Y_obs1 - Y_shift1) ** 2)

fig2, ax2 = plt.subplots(figsize=(13, 6))

# The same shape in two places: on axis, then moved to the image of the source
ax2.plot(Y_line1, psf_on_axis1, color='gray', linestyle='--', linewidth=2,
         label=r'on-axis source,  $\left| h(Y) \right|^2$')
ax2.plot(Y_line1, psf_shifted1, 'r-', label=r'source at $y_0$,  $\left| h(Y - Y_0) \right|^2$')
ax2.fill_between(Y_line1, 0, psf_shifted1, color='red', alpha=0.12)

# The shift itself
ax2.annotate("", xy=(Y_shift1, 0.78), xytext=(0.0, 0.78),
             arrowprops=dict(arrowstyle="<->", linewidth=1.8, color='black'))
ax2.text(Y_shift1 / 2, 0.82, r"$Y_0 = M y_0$", fontsize=16, ha='center')

# The separation that the PSF is actually evaluated at
ax2.plot([Y_obs1, Y_obs1], [0, h_sample1], color='green', linestyle=':',
         linewidth=1.8)
ax2.plot(Y_obs1, h_sample1, 'o', color='green', markersize=9, zorder=6)
ax2.annotate("", xy=(Y_obs1, 0.16), xytext=(Y_shift1, 0.16),
             arrowprops=dict(arrowstyle="<->", linewidth=1.8, color='green'))
# Centered a little left of the arrow's midpoint, so the subscript clears the
# dotted line rising from Y
ax2.text((Y_shift1 + Y_obs1) / 2 - 0.25, 0.195, r"$Y - Y_0$", fontsize=16,
         color='green', ha='center')
ax2.annotate(r"$\left| h(Y - Y_0) \right|^2 = %.3f$" % h_sample1,
             xy=(Y_obs1, h_sample1), xytext=(11.2, 0.60),
             arrowprops=dict(arrowstyle="->", color='green', linewidth=1.8),
             fontsize=15, color='green', ha='center')

# Mark the three positions on the axis itself. The labels sit just INSIDE the
# axes, below the baseline, so they cannot collide with the x tick labels.
for pos1, col1 in ((0.0, 'gray'), (Y_shift1, 'red'), (Y_obs1, 'green')):
    ax2.plot(pos1, 0, 'v', color=col1, markersize=9, zorder=6)
ax2.text(Y_shift1, -0.045, r"$Y_0$", color='red', fontsize=15, ha='center',
         va='top')
ax2.text(Y_obs1, -0.045, r"$Y$", color='green', fontsize=15, ha='center',
         va='top')

ax2.set_title("The PSF moves with the source but does not change shape",
              fontsize=17)
ax2.set_xlabel(r"image plane coordinate $Y$  ($\mu$m)")
ax2.set_ylabel(r"irradiance $\left| h \right|^2$")
ax2.set_xlim(-12, 16)
ax2.set_ylim(-0.13, 1.05)
ax2.legend(loc='upper right', frameon=True, edgecolor='black', fancybox=False,
           fontsize=13)
apply_custom_formatting(ax2)

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig2.savefig("../figures/ch05/ch05-17-psf-shift.jpg", dpi=200, bbox_inches='tight')
